<div style="
    position: relative;
    padding: 50px;
    background: linear-gradient(-45deg, #134e5e, #71b280, #00b09b, #96c93d);
    background-size: 400% 400%;
    animation: gradient 12s ease infinite;
    border-radius: 20px;
    color: white;
    font-family: 'Inter', sans-serif;
    box-shadow: 0 20px 40px rgba(0,0,0,0.15);
    overflow: hidden;
">
    <div style="
        position: absolute;
        top: 0; left: 0; right: 0; bottom: 0;
        background: rgba(0, 0, 0, 0.2);
        backdrop-filter: blur(8px);
        -webkit-backdrop-filter: blur(8px);
        border-radius: 20px;
        z-index: 1;
    "></div>

<div style="position: relative; z-index: 2;">
        <h1 style="font-size: 45px; font-weight: 900; margin: 0; color: #fff; text-shadow: 0 4px 10px rgba(0,0,0,0.2); border: none;">
            EXPLORATORY DATA ANALYSIS
        </h1>
        <p style="font-size: 25px; margin-top: 5px; color: #d4fc79; font-weight: 400;">
            🐦 BirdCLEF + 2026 competition
        </p>
        <div style="height: 4px; width: 100px; background: #fff; margin: 20px 0; border-radius: 2px;"></div>
</div>
</div>

<style>
    @keyframes gradient {
        0% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
        100% { background-position: 0% 50%; }
    }
</style>

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
    01. Import libraries & load the data
</h2>
</div>

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

from IPython.display import Audio

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="dark:#5A9_r"
)

plt.rcParams["figure.figsize"] = (10,6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["font.size"] = 11

In [ ]:
DATA_DIR = Path("/kaggle/input/competitions/birdclef-2026")

TRAIN_AUDIO = DATA_DIR / "train_audio"
TRAIN_SOUNDSCAPES = DATA_DIR / "train_soundscapes"
TEST_SOUNDSCAPES = DATA_DIR / "test_soundscapes"

TRAIN_META = DATA_DIR / "train.csv"
TAXONOMY = DATA_DIR / "taxonomy.csv"
SOUNDSCAPE_LABELS = DATA_DIR / "train_soundscapes_labels.csv"

print(DATA_DIR)

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        02. Dataset overview
</h2>
</div>

In [ ]:
train = pd.read_csv(TRAIN_META)
taxonomy = pd.read_csv(TAXONOMY)
soundscape_labels = pd.read_csv(SOUNDSCAPE_LABELS)

print(train.shape)
print(taxonomy.shape)
print(soundscape_labels.shape)

In [ ]:
train.head().style.background_gradient("Greens")

In [ ]:
len(train)

In [ ]:
train.collection.value_counts().to_frame("recordings").style.background_gradient("Greens")

In [ ]:
counts = train.collection.value_counts()

sns.barplot(
    x=counts.index,
    y=counts.values
)
for i,v in enumerate(counts.values):
    plt.text(i, v+10, str(v), ha="center")

plt.title("Recordings by Source Dataset")
plt.ylabel("Number of recordings")
plt.xlabel("")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        03. Class Distribution
</h2>
</div>

In [ ]:
species_counts = train.primary_label.value_counts()

species_counts.head(10).to_frame("recordings").style.background_gradient("Greens")

In [ ]:
species_counts.tail(10).to_frame("recordings").style.background_gradient("Greens")

In [ ]:
plt.figure()

sns.histplot(species_counts, bins=40)

plt.axvline(
    species_counts.median(),
    color="red",
    linestyle="--",
    label="median"
)

plt.xlabel("Recordings per species")
plt.title("Species Recording Distribution")
plt.legend()

plt.show()

In [ ]:
sns.histplot(species_counts, bins=50)
plt.yscale("log")
plt.title("Class imbalance (log scale)")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
    <h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        04. Recording Quality
    </h2>
</div>

In [ ]:
train.rating.describe()

In [ ]:
sns.histplot(train.rating, bins=20)
plt.title("Recording rating distribution")
plt.show()

In [ ]:
sns.boxplot(data=train, x="collection", y="rating")
plt.title("Recording quality by collection")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        05. Taxonomy overview
</h2>
</div>

In [ ]:
species_df = species_counts.reset_index()
species_df.columns = ["primary_label", "n_recordings"]

species_df = species_df.merge(
    taxonomy,
    on="primary_label",
    how="left"
)

In [ ]:
species_df.groupby("class_name").size()

In [ ]:
sns.countplot(data=species_df, x="class_name")
plt.title("Species per taxonomic class")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        06. Secondary labels
</h2>
</div>

In [ ]:
train["secondary_labels_list"] = train.secondary_labels.fillna("[]").apply(eval)

In [ ]:
train["n_secondary"] = train.secondary_labels_list.apply(len)

train.n_secondary.describe()

In [ ]:
sns.histplot(train.n_secondary)
plt.title("Number of secondary species")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        07. Geographic distribution
</h2>
</div>

In [ ]:
plt.figure()

plt.scatter(
    train.longitude,
    train.latitude,
    s=3,
    alpha=0.25
)

plt.title("Recording Locations")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [ ]:
train[["latitude","longitude"]].describe().style.background_gradient("Greens")

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        08. Soundscape labels
</h2>
</div>

In [ ]:
soundscape_labels.head().style.background_gradient("Greens")

In [ ]:
soundscape_labels.filename.nunique()

In [ ]:
soundscape_labels.groupby("filename").size().describe()

In [ ]:
soundscape_labels["species_count"] = soundscape_labels.primary_label.str.split(";").apply(len)

sns.histplot(soundscape_labels.species_count)
plt.title("Species per 5-second segment")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        09. Comparing species coverage
</h2>
</div>

In [ ]:
train_species = set(train.primary_label)

soundscape_species = set(
    soundscape_labels.primary_label
    .str.split(";")
    .explode()
)

In [ ]:
only_soundscape = soundscape_species - train_species
len(only_soundscape)

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        10. Inspecting audio files
</h2>
</div>

In [ ]:
row = train.sample(1).iloc[0]

audio_path = TRAIN_AUDIO / row.filename

print(row.primary_label)

In [ ]:
y, sr = librosa.load(audio_path, sr=32000)

print(len(y)/sr, "seconds")

In [ ]:
print("Species:", row.primary_label)
print("Rating:", row.rating)
print("Location:", row.latitude, row.longitude)

Audio(y, rate=sr)

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        11. Waveform
</h2>
</div>

In [ ]:
plt.figure()

librosa.display.waveshow(y, sr=sr)

plt.title(row.primary_label)
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        12. Spectrogram
</h2>
</div>

In [ ]:
mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128,
    fmax=16000
)

mel_db = librosa.power_to_db(mel)

In [ ]:
plt.figure(figsize=(12,4))

librosa.display.specshow(
    mel_db,
    sr=sr,
    x_axis="time",
    y_axis="mel",
    cmap="magma"
)

plt.colorbar(format="%+2.f dB")
plt.title(f"Mel Spectrogram – {row.primary_label}")

plt.tight_layout()
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        13. Inspecting several species
</h2>
</div>

In [ ]:
def show_random_species(label):

    row = train[train.primary_label == label].sample(1).iloc[0]
    path = TRAIN_AUDIO / row.filename
    
    y, sr = librosa.load(path, sr=32000)

    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_db = librosa.power_to_db(mel)

    plt.figure()
    librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel")
    plt.title(label)
    plt.show()

    display(Audio(y, rate=sr))

In [ ]:
show_random_species(train.primary_label.iloc[0])

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(14,6))

samples = train.sample(6)

for ax, (_,row) in zip(axes.flatten(), samples.iterrows()):

    y, sr = librosa.load(TRAIN_AUDIO / row.filename, sr=32000)

    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_db = librosa.power_to_db(mel)

    librosa.display.specshow(mel_db, sr=sr, ax=ax)

    ax.set_title(row.primary_label)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        14. Audio duration distribution
</h2>
</div>

In [ ]:
def get_duration(file):

    y, sr = librosa.load(file, sr=None)
    return len(y)/sr

In [ ]:
sample_files = train.sample(200)

durations = []

for f in sample_files.filename:

    path = TRAIN_AUDIO / f
    durations.append(get_duration(path))

In [ ]:
sns.histplot(durations, bins=30)

plt.xlabel("seconds")
plt.title("Audio duration distribution")
plt.show()

<div style="
    padding: 15px 25px;
    background: linear-gradient(90deg, #134e5e 0%, #71b280 100%);
    border-radius: 10px;
    border-left: 8px solid #d4fc79;
    color: white;
    font-family: 'Inter', sans-serif;
    margin-top: 40px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05);
">
<h2 style="margin: 0; font-size: 24px; font-weight: 700; letter-spacing: 0.5px; color: white; border: none;">
        15. Summary table
</h2>
</div>

In [ ]:
summary = pd.DataFrame({
    "metric":[
        "recordings",
        "species",
        "soundscapes labeled"
    ],
    "value":[
        len(train),
        train.primary_label.nunique(),
        soundscape_labels.filename.nunique()
    ]
})

summary.style.background_gradient("Greens")

<div style="
    padding: 30px; 
    background: linear-gradient(135deg, #134e5e 0%, #71b280 100%); 
    border-radius: 20px; 
    color: white; 
    text-align: center; 
    font-family: 'Inter', sans-serif;
    box-shadow: 0 10px 30px rgba(0,0,0,0.1);
    margin-top: 50px;
">
    <h2 style="color: white; margin: 0; font-size: 28px; border: none;">Thank you for checking this notebook!</h2>
    <p></p>
    <div style="
        display: inline-block; 
        padding: 12px 30px; 
        background-color: #d4fc79; 
        color: #134e5e; 
        border-radius: 50px; 
        font-weight: 800; 
        font-size: 18px;
        letter-spacing: 1px;
    ">
        ⭐ UPVOTE IF YOU LIKED IT!
    </div>
</div>